In [6]:
import cv2
import face_recognition
import os
import csv
from datetime import datetime

In [7]:
path = "imgs"
images = []
classNames = []

for file in os.listdir(path):
    img = face_recognition.load_image_file(f"{path}/{file}")
    images.append(img)
    classNames.append(os.path.splitext(file)[0])

In [8]:
encodings = []
def findEncodings(images):
    encodes = []
    for img in images:
        encode = face_recognition.face_encodings(img)[0]
        encodes.append(encode)
    return encodes

knownEncodings = findEncodings(images)

In [9]:
def markAttendance(name):
    today = datetime.now().strftime("%Y-%m-%d")
    time_now = datetime.now().strftime("%H:%M:%S")

    if not os.path.exists("attendance.csv"):
        with open("attendance.csv", "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Name", "Date", "Time"])

    with open("attendance.csv", "r+") as f:
        data = f.readlines()
        today_names = [line.split(',')[0] for line in data if today in line]

        if name not in today_names:
            f.write(f"{name},{today},{time_now}\n")
            print(f"Attendance marked for {name}")
        else:
            print(f"{name} already marked today")

In [10]:
Video = cv2.VideoCapture(0, cv2.CAP_DSHOW)

while True:
    ret, frame = Video.read()
    # if not ret:
    #     continue

    small = cv2.resize(frame, (0,0), fx=0.25, fy=0.25)
    rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

    faces = face_recognition.face_locations(rgb)
    encs = face_recognition.face_encodings(rgb, faces)

    for enc, loc in zip(encs, faces):
        matches = face_recognition.compare_faces(encodings, enc)
        if True in matches:
            name = names[matches.index(True)]

            if name not in marked:
                marked.add(name)
                with open("attendance.csv", "a", newline="") as f:
                    csv.writer(f).writerow([name, datetime.now().strftime("%H:%M:%S")])

            y1, x2, y2, x1 = [v*4 for v in loc]
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, name, (x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    cv2.imshow("Attendance", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

Video.release()
cv2.destroyAllWindows()